# LFC and FDR Calculation

**Kexin Dong**
**Apr 27, 2026**

This notebook quantifies cancer-associated mutation combinations amenable to modeling by PE7 multiplexed prime editing.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats
import seaborn as sns
import warnings
import os 
from pegg import prime
import matplotlib.patheffects as PathEffects
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
import re
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42   
mpl.rcParams['ps.fonttype'] = 42 
mpl.rcParams['text.usetex'] = False
from scipy.stats import combine_pvalues

yellows = sns.color_palette('Oranges').as_hex()
blues   = sns.color_palette('Blues').as_hex()
greens  = sns.color_palette('Greens').as_hex()
reds    = sns.color_palette('Reds').as_hex()
purples = sns.color_palette('Purples').as_hex()

## Overview and Rationale
The aim of this analysis is to show that our PE7 mouse model is a strong tool and has advantages in expanding the spectrum of cancer driver mutation combinations that we can model compared to traditional mouse models.

We will first use the MSK-IMPACT subset of AACR-GENIE data (we probably don't need an extremely large cohort — PEGG guide generation takes time to run and we don't want that to take forever; a reasonably sized pan-cancer cohort will be sufficient). We will retrieve a list of top 2 and top 3 high-VAF driver mutations from each patient in the cohort, generating a list of dual- and triple-mutation combinations that we would be interested in modeling in mice. We then refer to H2M Database v1 to convert them to mouse allele information to facilitate downstream analysis.

We will then look at how traditional mouse models, as well as all possible breeding efforts across them, help us cover these combinations. We will use the MMRRC database to get a list of available mouse models, and assess possible breeding by enumerating all possible combinations of any allele that we have ever had a mouse model for.

On the other hand, we quantify how many of these dual- and triple-mutation combinations can be modeled by PE7 multiplexed prime editing. There is no theoretical limit on the number or combination of mutations for multiplexed PE7 editing. As a result, whatever mutation we can design a pegRNA for via PEGG can be modeled as part of a mutational combination.

We compare and quantify these in a bar plot.

A potential issue to consider is whether to analyze mutations at the amino acid change level or the DNA change level.

## Data Sources
- AACR-GENIE
- MMRRC: www.mmrrc.org/
- PEGG pipeline: https://www.nature.com/articles/s41587-024-02172-9
- H2M: https://www.nature.com/articles/s41587-025-02925-0

## Step 1: Retrieve combination information from AACR-GENIE data

In [4]:
# stay consistent with H2M Database
df_mutation_aacr = pd.read_csv('/Users/kexindong/Documents/GitHub/Database/PublicDatabase/AACR-GENIE/v15.0/data_mutations_extended.txt',
                          header=0, sep='\t', comment="#", 
                          na_values=['Not Applicable', 'NA', 'NULL', '-', 'None', ''])
df_pts = pd.read_csv('/Users/kexindong/Documents/GitHub/Database/PublicDatabase/AACR-GENIE/v15.0/data_clinical_patient.txt', header=0, sep='\t', comment="#", na_values = 'Not Applicable')
df_sample = pd.read_csv('/Users/kexindong/Documents/GitHub/Database/PublicDatabase/AACR-GENIE/v15.0/data_clinical_sample.txt', header=0, sep='\t', comment="#", na_values = 'Not Applicable')

In [12]:
# subset to MSK patients
df_pts_impact = df_pts[df_pts['CENTER']=='MSK'].reset_index(drop=True)
df_pts_impact

,PATIENT_ID,SEX,PRIMARY_RACE,ETHNICITY,CENTER,INT_CONTACT,INT_DOD,YEAR_CONTACT,DEAD,YEAR_DEATH
0,GENIE-MSK-P-0000004,Female,White,Non-Spanish/non-Hispanic,MSK,14631,14631,2014,TRUE,2014
1,GENIE-MSK-P-0000015,Female,White,Non-Spanish/non-Hispanic,MSK,16656,16656,2015,TRUE,2015
2,GENIE-MSK-P-0000023,Male,White,Non-Spanish/non-Hispanic,MSK,22472,22472,2014,TRUE,2014
3,GENIE-MSK-P-0000024,Female,White,Non-Spanish/non-Hispanic,MSK,23469,23469,2016,TRUE,2016
4,GENIE-MSK-P-0000025,Female,White,Non-Spanish/non-Hispanic,MSK,27952,27952,2017,TRUE,2017
...,...,...,...,...,...,...,...,...,...,...
70402,GENIE-MSK-P-0084885,Female,White,Spanish/Hispanic,MSK,17601,NaN,2023,FALSE,NaN
70403,GENIE-MSK-P-0084888,Female,White,Non-Spanish/non-Hispanic,MSK,27810,NaN,2023,FALSE,NaN
70404,GENIE-MSK-P-0085089,Male,White,Non-Spanish/non-Hispanic,MSK,17370,NaN,2023,FALSE,NaN
70405,GENIE-MSK-P-0081132,Male,White,Non-Spanish/non-Hispanic,MSK,25093,NaN,2023,FALSE,NaN


In [13]:
pts_id_msk = set(df_pts_impact['PATIENT_ID'])
df_sample_impact = df_sample[df_sample['PATIENT_ID'].isin(pts_id_msk)].reset_index(drop=True)
df_sample_impact

,PATIENT_ID,SAMPLE_ID,AGE_AT_SEQ_REPORT,ONCOTREE_CODE,SAMPLE_TYPE,SEQ_ASSAY_ID,CANCER_TYPE,CANCER_TYPE_DETAILED,SAMPLE_TYPE_DETAILED
0,GENIE-MSK-P-0000004,GENIE-MSK-P-0000004-T01-IM3,39,IDC,Primary,MSK-IMPACT341,Breast Cancer,Breast Invasive Ductal Carcinoma,Primary tumor
1,GENIE-MSK-P-0000015,GENIE-MSK-P-0000015-T01-IM3,44,IDC,Metastasis,MSK-IMPACT341,Breast Cancer,Breast Invasive Ductal Carcinoma,Metastasis site unspecified
2,GENIE-MSK-P-0000023,GENIE-MSK-P-0000023-T01-IM3,61,PEMESO,Primary,MSK-IMPACT341,Mesothelioma,Peritoneal Mesothelioma,Primary tumor
3,GENIE-MSK-P-0000024,GENIE-MSK-P-0000024-T01-IM3,61,UEC,Metastasis,MSK-IMPACT341,Endometrial Cancer,Uterine Endometrioid Carcinoma,Metastasis site unspecified
4,GENIE-MSK-P-0000025,GENIE-MSK-P-0000025-T01-IM3,72,USC,Primary,MSK-IMPACT341,Endometrial Cancer,Uterine Serous Carcinoma/Uterine Papillary Ser...,Primary tumor
...,...,...,...,...,...,...,...,...,...
89975,GENIE-MSK-P-0084888,GENIE-MSK-P-0084888-T01-IH4,75,UNKNOWN,Unspecified,MSK-IMPACT-HEME-468,UNKNOWN,UNKNOWN,Not otherwise specified
89976,GENIE-MSK-P-0085089,GENIE-MSK-P-0085089-T01-IH4,46,MBN,Unspecified,MSK-IMPACT-HEME-468,Mature B-Cell Neoplasms,Mature B-Cell Neoplasms,Not otherwise specified
89977,GENIE-MSK-P-0085089,GENIE-MSK-P-0085089-T02-IH4,46,MBN,Unspecified,MSK-IMPACT-HEME-468,Mature B-Cell Neoplasms,Mature B-Cell Neoplasms,Not otherwise specified
89978,GENIE-MSK-P-0081132,GENIE-MSK-P-0081132-T01-IM7,67,PAAD,Primary,MSK-IMPACT505,Pancreatic Cancer,Pancreatic Adenocarcinoma,Primary tumor


In [14]:
df_mutation_impact = df_mutation_aacr[df_mutation_aacr['Tumor_Sample_Barcode'].isin(set(df_sample_impact['SAMPLE_ID']))].reset_index(drop=True)
df_mutation_impact

,Hugo_Symbol,Entrez_Gene_Id,Center,NCBI_Build,Chromosome,Start_Position,End_Position,Strand,Consequence,Variant_Classification,...,FILTER,Polyphen_Prediction,Polyphen_Score,SIFT_Prediction,SIFT_Score,SWISSPROT,n_depth,t_depth,Annotation_Status,mutationInCis_Flag
0,FAM46C,0.0,MSK,GRCh37,1,118165602,118165602,+,stop_gained,Nonsense_Mutation,...,PASS,NaN,NaN,NaN,NaN,NaN,414.0,601.0,SUCCESS,False
1,TEK,7010.0,MSK,GRCh37,9,27209136,27209136,+,missense_variant,Missense_Mutation,...,PASS,probably_damaging,0.964,deleterious,0.0,NaN,390.0,450.0,SUCCESS,False
2,DNMT3B,1789.0,MSK,GRCh37,20,31372564,31372564,+,"missense_variant,splice_region_variant",Missense_Mutation,...,PASS,benign,0.221,deleterious_low_confidence,0.0,NaN,543.0,585.0,SUCCESS,False
3,TMPRSS2,7113.0,MSK,GRCh37,21,42866465,42866465,+,missense_variant,Missense_Mutation,...,PASS,probably_damaging,0.913,deleterious,0.0,NaN,997.0,656.0,SUCCESS,False
4,TAP2,6891.0,MSK,GRCh37,6,32800190,32800190,+,stop_gained,Nonsense_Mutation,...,PASS,NaN,NaN,NaN,NaN,NaN,635.0,393.0,SUCCESS,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
651787,ARID5B,84159.0,MSK,GRCh37,10,63845618,63845618,+,frameshift_variant,Frame_Shift_Del,...,PASS,NaN,NaN,NaN,NaN,NaN,330.0,446.0,SUCCESS,False
651788,RNF43,54894.0,MSK,GRCh37,17,56432308,56432308,+,missense_variant,Missense_Mutation,...,PASS,probably_damaging,0.978,deleterious_low_confidence,0.0,NaN,507.0,539.0,SUCCESS,False
651789,CDK8,1024.0,MSK,GRCh37,13,26959350,26959350,+,"missense_variant,splice_region_variant",Missense_Mutation,...,PASS,probably_damaging,0.995,deleterious,0.0,NaN,289.0,417.0,SUCCESS,False
651790,EP300,2033.0,MSK,GRCh37,22,41553365,41553365,+,stop_gained,Nonsense_Mutation,...,PASS,NaN,NaN,NaN,NaN,NaN,773.0,306.0,SUCCESS,False


In [ ]:

# Pie plot: distribution of mutation types (Variant_Type)
variant_counts = df_mutation_impact['Variant_Type'].value_counts()

n = len(variant_counts)
cmap = plt.cm.Blues
colors = [cmap(0.3 + 0.6 * i / (n - 1)) for i in range(n)]

fig, ax = plt.subplots(figsize=(5, 5))
ax.axis('equal')
wedges, texts = ax.pie(
    variant_counts.values,
    labels=variant_counts.index,
    colors=colors,
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=1.5),
)
for t in texts:
    t.set_fontsize(13)
plt.setp(wedges, edgecolor='white')
fig.tight_layout()
plt.savefig('figures/variant_type_pie.pdf', bbox_inches='tight')
plt.show()

print(variant_counts.to_frame('Count'))


In [ ]:

# Pie plot: distribution of cancer types (CANCER_TYPE)
# merge small categories (<1%) into 'Other'
cancer_counts = df_sample_impact['CANCER_TYPE'].value_counts()
threshold = cancer_counts.sum() * 0.01
major = cancer_counts[cancer_counts >= threshold]
other_count = cancer_counts[cancer_counts < threshold].sum()
if other_count > 0:
    major['Other'] = other_count

n = len(major)
cmap = plt.cm.Greens
colors = [cmap(0.3 + 0.6 * i / max(n - 1, 1)) for i in range(n)]

fig, ax = plt.subplots(figsize=(7, 7))
ax.axis('equal')
wedges, texts = ax.pie(
    major.values,
    labels=major.index,
    colors=colors,
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=1.5),
)
for t in texts:
    t.set_fontsize(11)
plt.setp(wedges, edgecolor='white')
fig.tight_layout()
plt.savefig('figures/cancer_type_pie.pdf', bbox_inches='tight')
plt.show()

print(major.to_frame('Count'))


In [ ]:

# Bar plot: distribution of number of mutations per patient
muts_per_patient = (
    df_mutation_impact
    .groupby('Tumor_Sample_Barcode')
    .size()
    .reset_index(name='n_mutations')
    .merge(df_sample_impact[['SAMPLE_ID', 'PATIENT_ID']], left_on='Tumor_Sample_Barcode', right_on='SAMPLE_ID')
    .groupby('PATIENT_ID')['n_mutations'].sum()
)

bins = [0, 1, 2, 5, 10, 20, 50, 100, muts_per_patient.max() + 1]
labels = ['1', '2', '3–5', '6–10', '11–20', '21–50', '51–100', '>100']
binned = pd.cut(muts_per_patient, bins=bins, labels=labels, right=True)
bin_counts = binned.value_counts().reindex(labels)

n = len(bin_counts)
cmap = plt.cm.Blues
colors = [cmap(0.35 + 0.55 * i / (n - 1)) for i in range(n)]

fig, ax = plt.subplots(figsize=(8, 6))
ax.bar(range(n), bin_counts.values, color=colors, edgecolor='black', linewidth=1, alpha=0.85)
ax.set_xticks(range(n))
ax.set_xticklabels(labels, fontsize=14)
ax.set_xlabel('No. of Mutations per Patient', fontsize=16)
ax.set_ylabel('No. of Patients', fontsize=16)
ax.tick_params(axis='both', which='major', labelsize=14)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.savefig('figures/mutations_per_patient.pdf', bbox_inches='tight')
plt.show()
